In [0]:
dbutils.library.restartPython()


In [0]:
dbutils.widgets.text("path", "")
config_path =dbutils.widgets.get("path") 

In [0]:

from common_utils.logging import get_logger
import json 

# =========================================
# 0.Logger
# =========================================
logger = get_logger('Github_ingestion.ipynb')
logger.info('Session started')

# =========================================
# 1. CONFIG PATH
# =========================================
dbutils.widgets.text("path", "")
config_path =dbutils.widgets.get("path") 
print(config_path)
# =========================================
# 2. Read Config
# =========================================
with open(config_path,"r") as f:
    config = json.load(f)


# =========================================
# 3. CONFIG Sections
# =========================================
source_config = config['source']
target_config = config['target']
load_config = config['load_options']

# =========================================
# 4. Run Date
# =========================================

run_date = load_config['load_date']


# =========================================
# 5. source and target paths
# =========================================

source_path = source_config["source_path"]
target_path = (
    f"{target_config['volume_path']}/"
    f"load_date={run_date}"
)

logger.info(f"Source path: {source_path}")
logger.info(f"Target path: {target_path}")

# =========================================
# 6. List source files
# =========================================

source_files = [
    file for file in dbutils.fs.ls(source_path)
    if not file.isDir()
    and file.name.lower().endswith(".csv")

]

if len(source_files) != 7:
    raise ValueError(
        f"Expected 7 CSV files, found {len(source_files)}"
    )


# =========================================
# 7. Create target directory
# =========================================

dbutils.fs.mkdirs(target_path)



# =========================================
# 8. Copy all the files to the volumn
# =========================================

for file in source_files:
    destination = f"{target_path}/{file.name}"

    logger.info("Copying  {file.name}")

    dbutils.fs.cp(
        file.path,
        destination
    )

    logger.info(f"Successfully copied {file.name}")



# =========================================
#9. verify ingestion
# =========================================

landed_files = dbutils.fs.ls(target_path)

logger.info(
    f"Total files landed: {len(landed_files)}"
)

logger.info("GitHub ingestion completed")


